# Zeitfenster für einen wöchentlichen Check-up

Ein Coach soll während des Kurses einmal pro Woche Hinweise auf mögliche
spätere Abbrüche oder ein Nichtbestehen erhalten.

Für mehrere Stichtage prüfen wir zunächst:
- Wie viele Kurseinträge sind noch angemeldet?
- Wie viele davon enden später mit Abbruch oder Nichtbestehen?
- Wie viele spätere Abmeldungen erfolgen bereits innerhalb von 14 Tagen?

Diese Übersicht bewertet noch kein Modell. Sie hilft uns, einen sinnvollen
Beginn und ein sinnvolles Ende der wöchentlichen Prüfung festzulegen.

In [1]:
from pathlib import Path

import pandas as pd

# Passe nur diesen Pfad an, falls dein Notebook nicht im Projektordner liegt.
rohordner = Path("Daten OULAD")
schluessel = ["code_module", "code_presentation", "id_student"]

# Wir lesen nur die drei kleinen Tabellen, die für die Zeitübersicht nötig sind.
info = pd.read_csv(
    rohordner / "studentInfo.csv",
    usecols=schluessel + ["final_result"],
)
registrierung = pd.read_csv(
    rohordner / "studentRegistration.csv",
    usecols=schluessel + ["date_registration", "date_unregistration"],
)
kurse = pd.read_csv(rohordner / "courses.csv")

# Je nach CSV-Version heißt die Spalte für die Kursdauer unterschiedlich.
dauerspalte = (
    "module_presentation_length"
    if "module_presentation_length" in kurse.columns
    else "length"
)
kurse = kurse[
    ["code_module", "code_presentation", dauerspalte]
].rename(columns={dauerspalte: "kursdauer"})

# Die Tabellen werden zu einem Eintrag pro Person und Kurs verbunden.
basis = (
    info.merge(
        registrierung,
        on=schluessel,
        validate="one_to_one",
    )
    .merge(
        kurse,
        on=["code_module", "code_presentation"],
        validate="many_to_one",
    )
)

# Das Fragezeichen in den Rohdaten wird zu einem fehlenden Zahlenwert.
basis["anmeldetag"] = pd.to_numeric(
    basis["date_registration"], errors="coerce"
)
basis["abmeldetag"] = pd.to_numeric(
    basis["date_unregistration"], errors="coerce"
)
basis["kursdauer"] = pd.to_numeric(
    basis["kursdauer"], errors="coerce"
)

# Bei "Withdrawn" ohne lesbaren Abmeldetag kennen wir den Zeitpunkt nicht.
# Diese Fälle lassen sich keinem Stichtag zuverlässig zuordnen.
abbruch_ohne_datum = (
    basis["final_result"].eq("Withdrawn")
    & basis["abmeldetag"].isna()
)
print("Abbruch ohne lesbaren Abmeldetag:", abbruch_ohne_datum.sum())
basis = basis.loc[~abbruch_ohne_datum].copy()

# OULAD beginnt bei Tag 0: Tag 6 ist das Ende der ersten Kurswoche.
# Tag 28 nehmen wir zusätzlich auf, um unser bestehendes Modell einzuordnen.
stichtage = [6, 13, 20, 27, 28, 41, 55, 83, 111, 167, 209]
ergebnisse = []

for tag in stichtage:
    # Der Coach sieht nur Personen, die bis dahin angemeldet wurden,
    # noch nicht abgemeldet sind und deren Kurs noch läuft.
    aktiv = basis.loc[
        basis["anmeldetag"].le(tag)
        & (basis["abmeldetag"].isna() | basis["abmeldetag"].gt(tag))
        & basis["kursdauer"].gt(tag)
    ]

    spaeterer_abbruch = aktiv["final_result"].eq("Withdrawn")
    nichtbestehen = aktiv["final_result"].eq("Fail")
    faelle = spaeterer_abbruch.sum() + nichtbestehen.sum()

    # Diese Abmeldungen stehen kurz bevor; dafür wäre wenig Vorlauf da.
    abbruch_in_14_tagen = (
        spaeterer_abbruch
        & aktiv["abmeldetag"].le(tag + 14)
    ).sum()

    ergebnisse.append({
        "stichtag": tag,
        "noch_aktiv": len(aktiv),
        "spaeterer_abbruch": int(spaeterer_abbruch.sum()),
        "spaeteres_nichtbestehen": int(nichtbestehen.sum()),
        "zielanteil_prozent": round(100 * faelle / len(aktiv), 1),
        "abbruch_in_naechsten_14_tagen": int(abbruch_in_14_tagen),
        "median_vorlauf_abbruch_tage": round(
            (aktiv.loc[spaeterer_abbruch, "abmeldetag"] - tag).median(),
            1,
        ),
    })

zeitfenster = pd.DataFrame(ergebnisse)
zeitfenster

Abbruch ohne lesbaren Abmeldetag: 93


,stichtag,noch_aktiv,spaeterer_abbruch,spaeteres_nichtbestehen,zielanteil_prozent,abbruch_in_naechsten_14_tagen,median_vorlauf_abbruch_tage
0,6,29002,6650,7008,47.1,1280,73.0
1,13,27970,5589,7021,45.1,571,87.0
2,20,27787,5383,7031,44.7,663,82.0
3,27,27435,5025,7032,43.9,519,82.0
4,28,27422,5012,7032,43.9,514,81.0
5,41,26922,4510,7033,42.9,502,79.0
6,55,26423,4008,7034,41.8,450,75.0
7,83,25629,3209,7037,40.0,375,65.0
8,111,24839,2417,7038,38.1,317,52.0
9,167,23469,1047,7038,34.4,299,34.0


## Ab wann sind Assessment-Informationen verfügbar?

Für jeden Wochenstichtag prüfen wir, bei wie vielen noch angemeldeten
Kurseinträgen bereits mindestens ein Assessment fällig war.

Ein fälliges Assessment bedeutet noch nicht, dass die Person abgegeben
oder nicht abgegeben hat. Die Auswertung zeigt zunächst nur, ab wann
Assessment-Merkmale für unsere wöchentlichen Prognosen möglich sind.

In [3]:
# Wir lesen den Assessment-Plan ein. Individuelle Abgaben und die
# große VLE-Datei werden für diese Übersicht noch nicht benötigt.
assessments = pd.read_csv(
    rohordner / "assessments.csv",
    usecols=[
        "code_module",
        "code_presentation",
        "assessment_type",
        "date",
    ],
)

# Nicht lesbare Fälligkeitstage werden als fehlend behandelt.
assessments["faellig_am_tag"] = pd.to_numeric(
    assessments["date"],
    errors="coerce",
)

# Pro Kurspräsentation ermitteln wir den ersten bekannten Termin
# einer regulären Aufgabe (TMA oder CMA).
erste_faelligkeit = (
    assessments.loc[
        assessments["assessment_type"].isin(["TMA", "CMA"])
        & assessments["faellig_am_tag"].ge(0)
    ]
    .groupby(
        ["code_module", "code_presentation"],
        as_index=False,
    )["faellig_am_tag"]
    .min()
    .rename(columns={"faellig_am_tag": "erste_faelligkeit"})
)

# Wir ergänzen jeden Kurseintrag um den ersten Termin seines Kurses.
basis_mit_termin = basis.merge(
    erste_faelligkeit,
    on=["code_module", "code_presentation"],
    how="left",
    validate="many_to_one",
)

ergebnisse_faelligkeit = []

for tag in stichtage:
    # Dieselbe Gruppe noch angemeldeter Personen wie in der
    # vorherigen Zeitfenster-Auswertung.
    aktiv = basis_mit_termin.loc[
        basis_mit_termin["anmeldetag"].le(tag)
        & (
            basis_mit_termin["abmeldetag"].isna()
            | basis_mit_termin["abmeldetag"].gt(tag)
        )
        & basis_mit_termin["kursdauer"].gt(tag)
    ]

    mit_faelligem_assessment = aktiv["erste_faelligkeit"].le(tag).sum()

    ergebnisse_faelligkeit.append({
        "stichtag": tag,
        "noch_aktiv": len(aktiv),
        "mit_faelligem_assessment": int(mit_faelligem_assessment),
        "anteil_prozent": round(
            100 * mit_faelligem_assessment / len(aktiv),
            1,
        ),
    })

faelligkeit_nach_woche = pd.DataFrame(ergebnisse_faelligkeit)
faelligkeit_nach_woche

,stichtag,noch_aktiv,mit_faelligem_assessment,anteil_prozent
0,6,29002,0,0.0
1,13,27970,1287,4.6
2,20,27787,15744,56.7
3,27,27435,22453,81.8
4,28,27422,22441,81.8
5,41,26922,24499,91.0
6,55,26423,24019,90.9
7,83,25629,25629,100.0
8,111,24839,24839,100.0
9,167,23469,23469,100.0


## Ab wann liegen eigene Abgaben vor?

Wir zählen für jeden Stichtag Personen mit mindestens einer bis dahin
erfassten eigenen Abgabe. Übertragene Leistungen zählen hier nicht als
eigene Abgabe. „Fällig ohne eigene Abgabe“ ist noch kein Versäumnis:
Eine Leistung kann beispielsweise angerechnet worden sein.

In [4]:
# Assessment-ID mit dem zugehörigen Kurs verbinden.
assessment_zuordnung = pd.read_csv(
    rohordner / "assessments.csv",
    usecols=[
        "id_assessment",
        "code_module",
        "code_presentation",
        "assessment_type",
    ],
)

# Wir benötigen nur den Einreichtag und die Information,
# ob eine Leistung aus einem früheren Durchlauf stammt.
abgaben = pd.read_csv(
    rohordner / "studentAssessment.csv",
    usecols=[
        "id_assessment",
        "id_student",
        "date_submitted",
        "is_banked",
    ],
)

abgaben["date_submitted"] = pd.to_numeric(
    abgaben["date_submitted"], errors="coerce"
)
abgaben["is_banked"] = pd.to_numeric(
    abgaben["is_banked"], errors="coerce"
)

abgaben = abgaben.merge(
    assessment_zuordnung,
    on="id_assessment",
    validate="many_to_one",
)

# Pro Kurseintrag merken wir uns die erste eigene Abgabe.
erste_eigene_abgabe = (
    abgaben.loc[
        abgaben["assessment_type"].isin(["TMA", "CMA"])
        & abgaben["is_banked"].eq(0)
        & abgaben["date_submitted"].ge(0)
    ]
    .groupby(schluessel, as_index=False)["date_submitted"]
    .min()
    .rename(columns={
        "date_submitted": "erste_eigene_abgabe"
    })
)

basis_mit_abgabe = basis_mit_termin.merge(
    erste_eigene_abgabe,
    on=schluessel,
    how="left",
    validate="one_to_one",
)

ergebnisse_abgaben = []

for tag in stichtage:
    # Wieder exakt dieselbe Gruppe noch angemeldeter Personen.
    aktiv = basis_mit_abgabe.loc[
        basis_mit_abgabe["anmeldetag"].le(tag)
        & (
            basis_mit_abgabe["abmeldetag"].isna()
            | basis_mit_abgabe["abmeldetag"].gt(tag)
        )
        & basis_mit_abgabe["kursdauer"].gt(tag)
    ]

    faellig = aktiv["erste_faelligkeit"].le(tag)
    eigene_abgabe = aktiv["erste_eigene_abgabe"].le(tag)

    ergebnisse_abgaben.append({
        "stichtag": tag,
        "noch_aktiv": len(aktiv),
        "mit_eigener_abgabe": int(eigene_abgabe.sum()),
        "anteil_mit_abgabe_prozent": round(
            100 * eigene_abgabe.mean(), 1
        ),
        "faellig_ohne_eigene_abgabe": int(
            (faellig & ~eigene_abgabe).sum()
        ),
    })

abgaben_nach_woche = pd.DataFrame(ergebnisse_abgaben)
abgaben_nach_woche

,stichtag,noch_aktiv,mit_eigener_abgabe,anteil_mit_abgabe_prozent,faellig_ohne_eigene_abgabe
0,6,29002,655,2.3,0
1,13,27970,2875,10.3,263
2,20,27787,12846,46.2,4322
3,27,27435,19702,71.8,2945
4,28,27422,19829,72.3,2874
5,41,26922,22242,82.6,2384
6,55,26423,22710,85.9,1996
7,83,25629,23566,92.0,2063
8,111,24839,22995,92.6,1844
9,167,23469,21894,93.3,1575


## Modellmerkmale für beliebige Wochenstichtage

Der bisherige Tag-28-Datensatz dient als Kontrollpunkt. Wir berechnen
zunächst seine Assessment-Merkmale erneut, diesmal mit einer Funktion,
die auch andere Stichtage akzeptiert.

Eine Abgabe zählt nur, wenn sie bis zum jeweiligen Stichtag erfasst war.
Angerechnete Leistungen werden bei fälligen Assessments als erfüllt
berücksichtigt. Assessment-Noten verwenden wir nicht.

In [5]:
# Der Assessment-Plan liefert Kurszuordnung und Fälligkeit jeder Aufgabe.
plan = pd.read_csv(
    rohordner / "assessments.csv",
    usecols=[
        "id_assessment",
        "code_module",
        "code_presentation",
        "assessment_type",
        "date",
    ],
)

plan["date"] = pd.to_numeric(plan["date"], errors="coerce")
plan = plan.loc[
    plan["assessment_type"].isin(["TMA", "CMA"])
].copy()

assert plan["id_assessment"].is_unique

# Einreichungen werden über die Assessment-ID ihrem Kurs zugeordnet.
abgaben_modell = pd.read_csv(
    rohordner / "studentAssessment.csv",
    usecols=[
        "id_assessment",
        "id_student",
        "date_submitted",
        "is_banked",
    ],
)

abgaben_modell["date_submitted"] = pd.to_numeric(
    abgaben_modell["date_submitted"], errors="coerce"
)
abgaben_modell["is_banked"] = pd.to_numeric(
    abgaben_modell["is_banked"], errors="coerce"
)

abgaben_modell = abgaben_modell.merge(
    plan[
        [
            "id_assessment",
            "code_module",
            "code_presentation",
            "date",
        ]
    ],
    on="id_assessment",
    how="inner",
    validate="many_to_one",
)

print("Reguläre Assessments:", len(plan))
print("Zugeordnete Einreichungen:", len(abgaben_modell))

Reguläre Assessments: 182
Zugeordnete Einreichungen: 168953


In [6]:
kurs = ["code_module", "code_presentation"]

def zaehle_assessments(tabelle, gruppen, spaltenname):
    """Zählt verschiedene Assessments pro Kurs oder Kurseintrag."""
    return (
        tabelle.groupby(gruppen, as_index=False)["id_assessment"]
        .nunique()
        .rename(columns={"id_assessment": spaltenname})
    )


def assessment_snapshot(tag):
    """Erstellt die bis zum angegebenen Tag bekannten Assessment-Merkmale."""

    # Dieselbe Einschlussregel wie in unserer Zeitfenster-Auswertung.
    aktuell = basis.loc[
        basis["anmeldetag"].le(tag)
        & (
            basis["abmeldetag"].isna()
            | basis["abmeldetag"].gt(tag)
        )
        & basis["kursdauer"].gt(tag),
        schluessel + ["anmeldetag", "final_result"],
    ].copy()

    aktuell["stichtag"] = tag
    aktuell["abbruch_oder_fail"] = aktuell["final_result"].isin(
        ["Withdrawn", "Fail"]
    )
    aktuell = aktuell.drop(columns="final_result")

    # Ein Assessment zählt nur, wenn sein Termin von Tag 0
    # bis einschließlich des Stichtags liegt.
    faellig = plan.loc[plan["date"].between(0, tag)]
    anzahl_faellig = zaehle_assessments(
        faellig, kurs, "assessments_faellig"
    )

    # Eigene Abgaben: bis zum Stichtag eingereicht.
    # Übertragene Leistungen: bis dahin im Datensatz eingetragen.
    eigene_sichtbar = (
        abgaben_modell["is_banked"].eq(0)
        & abgaben_modell["date_submitted"].between(0, tag)
    )
    banked_sichtbar = (
        abgaben_modell["is_banked"].eq(1)
        & abgaben_modell["date_submitted"].le(tag)
    )
    sichtbar = abgaben_modell.loc[
        eigene_sichtbar | banked_sichtbar
    ]

    anzahl_eigene = zaehle_assessments(
        sichtbar.loc[sichtbar["is_banked"].eq(0)],
        schluessel,
        "abgaben",
    )

    # Als erfüllt zählen nur bereits fällige Aufgaben.
    erfuellt = sichtbar.loc[
        sichtbar["id_assessment"].isin(faellig["id_assessment"])
    ]
    anzahl_erfuellt = zaehle_assessments(
        erfuellt, schluessel, "assessments_erfuellt"
    )
    anzahl_banked = zaehle_assessments(
        erfuellt.loc[erfuellt["is_banked"].eq(1)],
        schluessel,
        "banked_faellige",
    )

    aktuell = aktuell.merge(
        anzahl_faellig,
        on=kurs,
        how="left",
        validate="many_to_one",
    )

    for merkmale in [
        anzahl_eigene,
        anzahl_erfuellt,
        anzahl_banked,
    ]:
        aktuell = aktuell.merge(
            merkmale,
            on=schluessel,
            how="left",
            validate="one_to_one",
        )

    anzahlspalten = [
        "assessments_faellig",
        "abgaben",
        "assessments_erfuellt",
        "banked_faellige",
    ]
    aktuell[anzahlspalten] = (
        aktuell[anzahlspalten].fillna(0).astype(int)
    )

    aktuell["assessments_fehlend"] = (
        aktuell["assessments_faellig"]
        - aktuell["assessments_erfuellt"]
    )

    assert not aktuell.duplicated(schluessel).any()
    assert aktuell["assessments_fehlend"].ge(0).all()

    return aktuell

In [7]:
probe_tag28 = assessment_snapshot(28)

print("Kurseinträge:", len(probe_tag28))
print(
    "Abbruch oder Fail:",
    probe_tag28["abbruch_oder_fail"].sum(),
)
print(
    "Mit fälligem Assessment:",
    probe_tag28["assessments_faellig"].gt(0).sum(),
)
print(
    "Mit eigener Abgabe:",
    probe_tag28["abgaben"].gt(0).sum(),
)
print("Fehlende Werte:", probe_tag28.isna().sum().sum())

Kurseinträge: 27422
Abbruch oder Fail: 12044
Mit fälligem Assessment: 22441
Mit eigener Abgabe: 19829
Fehlende Werte: 0


## VLE-Aktivität für wöchentliche Datenstände

Die VLE-Datei wird einmal eingelesen. Für jeden Kurseintrag zählen wir
die Klicks je Kurswoche (Tag 0–6, 7–13 usw.).

Ein wöchentlicher Datenstand enthält ausschließlich Klicks bis zu seinem
Stichtag: die bisherige Gesamtzahl und die Zahl der letzten sieben Tage.

In [8]:
# Alle Werte sind Enden vollständiger Kurswochen.
modell_stichtage = [6, 13, 20, 27, 55, 83, 111, 167, 209]
assert all(tag % 7 == 6 for tag in modell_stichtage)

wochen_teile = []
maximaler_tag = max(modell_stichtage)

for nummer, block in enumerate(
    pd.read_csv(
        rohordner / "studentVle.csv",
        usecols=schluessel + ["date", "sum_click"],
        chunksize=400_000,
    ),
    start=1,
):
    # Aktivitäten vor Kursbeginn und nach unserem letzten
    # Vergleichstag gehören nicht in diese Auswertung.
    block["date"] = pd.to_numeric(block["date"], errors="coerce")
    block = block.loc[
        block["date"].between(0, maximaler_tag)
    ].copy()

    if block.empty:
        continue

    block["sum_click"] = pd.to_numeric(
        block["sum_click"], errors="coerce"
    )
    assert block["sum_click"].notna().all()

    # Woche 0 umfasst Tag 0–6, Woche 1 Tag 7–13 usw.
    block["woche"] = (block["date"] // 7).astype(int)

    wochen_teile.append(
        block.groupby(
            schluessel + ["woche"]
        )["sum_click"].sum()
    )

    if nummer % 10 == 0:
        print("CSV-Blöcke verarbeitet:", nummer)

# Teilsummen derselben Person, Kurspräsentation und Woche addieren.
wochenklicks = (
    pd.concat(wochen_teile)
    .groupby(level=schluessel + ["woche"])
    .sum()
    .reset_index()
)

# Eine Zeile je Kurseintrag und eine Spalte je Kurswoche.
klicks_pro_woche = (
    wochenklicks
    .pivot(
        index=schluessel,
        columns="woche",
        values="sum_click",
    )
    .reindex(
        columns=range(maximaler_tag // 7 + 1),
        fill_value=0,
    )
    .fillna(0)
    .astype(int)
)

# Aufsummierte Klicks bis zum Ende jeder Woche.
klicks_kumuliert = klicks_pro_woche.cumsum(axis=1)

print("Kurseinträge mit VLE-Aktivität:", len(klicks_pro_woche))
print("Erfasste Kurswochen:", klicks_pro_woche.shape[1])

CSV-Blöcke verarbeitet: 10
CSV-Blöcke verarbeitet: 20
Kurseinträge mit VLE-Aktivität: 28494
Erfasste Kurswochen: 30


In [9]:
def baue_wochensnapshot(tag):
    """Erstellt einen vollständigen Datenstand am Ende einer Kurswoche."""
    assert tag % 7 == 6, "Der Stichtag muss ein Wochenendtag sein."

    woche = tag // 7

    # Die Assessment-Funktion enthält bereits die Personen,
    # die am Stichtag noch zum Kurs gehören.
    snapshot = assessment_snapshot(tag)

    vle_merkmale = pd.DataFrame({
        "klicks_bis_stichtag": klicks_kumuliert[woche],
        "klicks_letzte_7_tage": klicks_pro_woche[woche],
    }).reset_index()

    snapshot = snapshot.merge(
        vle_merkmale,
        on=schluessel,
        how="left",
        validate="one_to_one",
    )

    # Kein VLE-Eintrag bedeutet hier null beobachtete Klicks.
    klickspalten = [
        "klicks_bis_stichtag",
        "klicks_letzte_7_tage",
    ]
    snapshot[klickspalten] = (
        snapshot[klickspalten].fillna(0).astype(int)
    )

    assert not snapshot.duplicated(schluessel).any()
    assert not snapshot.isna().any().any()

    return snapshot


# Zuerst einen Stichtag prüfen, dann alle Wochen erzeugen.
probe_tag27 = baue_wochensnapshot(27)

print("Kurseinträge:", len(probe_tag27))
print("Abbruch oder Fail:", probe_tag27["abbruch_oder_fail"].sum())
print("Fehlende Werte:", probe_tag27.isna().sum().sum())
display(probe_tag27.head())

Kurseinträge: 27435
Abbruch oder Fail: 12057
Fehlende Werte: 0


,code_module,code_presentation,id_student,anmeldetag,stichtag,abbruch_oder_fail,assessments_faellig,abgaben,assessments_erfuellt,banked_faellige,assessments_fehlend,klicks_bis_stichtag,klicks_letzte_7_tage
0,AAA,2013J,11391,-159.0,27,False,1,1,1,0,0,303,0
1,AAA,2013J,28400,-53.0,27,False,1,1,1,0,0,335,57
2,AAA,2013J,31604,-52.0,27,False,1,1,1,0,0,309,25
3,AAA,2013J,32885,-176.0,27,False,1,1,1,0,0,272,58
4,AAA,2013J,38053,-110.0,27,False,1,1,1,0,0,450,39


## Wochenweise Modellbasis

Wir prüfen die Klicks der ersten Kurswoche gegen den bestehenden
Tag-28-Datensatz. Anschließend erstellen wir für jeden ausgewählten
Wochenstichtag einen Datenstand. Derselbe Kurseintrag darf dabei in
mehreren Wochen vorkommen; bei der späteren Modellprüfung trennen wir
Personen zwischen Training und Validierung.

In [10]:
# Der Datenordner liegt direkt im Projektordner.
projektordner = rohordner.resolve().parent
datei_tag28 = (
    projektordner
    / "data"
    / "processed"
    / "oulad_tag28_modellbasis.csv"
)

alt_tag28 = pd.read_csv(
    datei_tag28,
    usecols=schluessel + ["klicks_tag_0_bis_6"],
)

neu_woche0 = (
    klicks_pro_woche[0]
    .rename("klicks_woche0_neu")
    .reset_index()
)

kontrolle = alt_tag28.merge(
    neu_woche0,
    on=schluessel,
    how="left",
    validate="one_to_one",
)

# Ohne beobachtete VLE-Aktivität sind die Klicks null.
kontrolle["klicks_woche0_neu"] = (
    kontrolle["klicks_woche0_neu"].fillna(0).astype(int)
)

abweichungen = (
    kontrolle["klicks_tag_0_bis_6"]
    != kontrolle["klicks_woche0_neu"]
).sum()

print("Verglichene Kurseinträge:", len(kontrolle))
print("Abweichungen bei Klicks an Tag 0–6:", abweichungen)

assert abweichungen == 0

Verglichene Kurseinträge: 27422
Abweichungen bei Klicks an Tag 0–6: 0


In [11]:
# Die große VLE-Datei wird hierfür nicht erneut eingelesen:
# baue_wochensnapshot nutzt unsere bereits berechneten Wochenklicks.
wochenbasis = pd.concat(
    [
        baue_wochensnapshot(tag)
        for tag in modell_stichtage
    ],
    ignore_index=True,
)

# Ein Kurseintrag darf je Stichtag genau einmal vorkommen.
assert not wochenbasis.duplicated(
    ["stichtag"] + schluessel
).any()
assert wochenbasis.isna().sum().sum() == 0

uebersicht_wochenbasis = (
    wochenbasis.groupby("stichtag")
    .agg(
        kurseintraege=("id_student", "size"),
        faelle=("abbruch_oder_fail", "sum"),
    )
)

display(uebersicht_wochenbasis)

zielordner = projektordner / "data" / "processed"
zielordner.mkdir(parents=True, exist_ok=True)
datei_wochen = zielordner / "oulad_wochenmodellbasis.csv"

wochenbasis.to_csv(datei_wochen, index=False)

print("Gespeichert:", datei_wochen)
print("Form:", wochenbasis.shape)
print("Fehlende Werte:", wochenbasis.isna().sum().sum())

,kurseintraege,faelle
stichtag,,
6,29002,13658
13,27970,12610
20,27787,12414
27,27435,12057
55,26423,11042
83,25629,10246
111,24839,9455
167,23469,8085
209,22812,7428


Gespeichert: C:\Users\Tony Willert\OneDrive\Weiterbildung\PortfolioProjekt\data\processed\oulad_wochenmodellbasis.csv
Form: (235366, 13)
Fehlende Werte: 0


## Wöchentliche Datenstände für den Prototyp

Das vorläufige Arbeitsfenster reicht vom Ende der ersten Kurswoche
(Tag 6) bis zum Ende der sechzehnten Kurswoche (Tag 111).

Für jede dieser Wochen wird ein Datenstand mit ausschließlich bis dahin
verfügbaren Assessment- und VLE-Merkmalen erzeugt. Die Auswahl des
Fensters ist vorläufig und wird bei der Auswertung wiederholter Hinweise
überprüft.

In [12]:
# Tag 6, 13, 20, ... bis einschließlich Tag 111.
woechentliche_stichtage = list(range(6, 112, 7))
assert len(woechentliche_stichtage) == 16
assert woechentliche_stichtage[-1] == 111

# Die Funktionen und VLE-Wochensummen sind bereits im Notebook vorhanden.
woechentliche_basis = pd.concat(
    [
        baue_wochensnapshot(tag)
        for tag in woechentliche_stichtage
    ],
    ignore_index=True,
)

# Dieselbe Person darf in verschiedenen Wochen vorkommen,
# aber innerhalb eines Stichtags nur einmal pro Kurseintrag.
assert not woechentliche_basis.duplicated(
    ["stichtag"] + schluessel
).any()
assert woechentliche_basis.isna().sum().sum() == 0

zielordner = projektordner / "data" / "processed"
datei_woechentlich = (
    zielordner / "oulad_wochenmodellbasis_tag6_bis111.csv"
)
woechentliche_basis.to_csv(
    datei_woechentlich,
    index=False,
)

print("Stichtage:", woechentliche_stichtage)
print("Form:", woechentliche_basis.shape)
print("Fehlende Werte:", woechentliche_basis.isna().sum().sum())
print("Gespeichert:", datei_woechentlich)

Stichtage: [6, 13, 20, 27, 34, 41, 48, 55, 62, 69, 76, 83, 90, 97, 104, 111]
Form: (423510, 13)
Fehlende Werte: 0
Gespeichert: C:\Users\Tony Willert\OneDrive\Weiterbildung\PortfolioProjekt\data\processed\oulad_wochenmodellbasis_tag6_bis111.csv
